## 1. Tải numerics đầy đủ
- Đầu vào: `screening_checkpoint.parquet` (danh sách bệnh nhân đã sàng lọc).
- Đầu ra: `data/raw_numerics/<record>.parquet` (SPO2, PULSE, HR, RESP, ABPMEAN) và `data/download_manifest.parquet`.
- Giá trị 0 trong numerics là mất tín hiệu, nên được đổi thành NaN.
- Chọn `N_PATIENTS` bệnh nhân **ngẫu nhiên** (`SEED`) trong số bệnh nhân đạt sàng lọc, không lấy N dòng đầu file (D5). Dữ liệu hiện có được tải bằng cách cũ (`head`), nên nếu chạy lại cell tải, notebook sẽ tải thêm các bệnh nhân mới được chọn; file đã tải không bị xóa và các bước sau dùng mọi record có trong manifest.
- Có resume: chạy lại cell sẽ tiếp tục từ chỗ dừng.

In [ ]:
import os, time, socket
import numpy as np, pandas as pd, requests, wfdb
from tqdm.auto import tqdm

DB_NAME, DB_VERSION = "mimic3wdb-matched", "1.0"
RECORDS_URL = f"https://physionet.org/files/{DB_NAME}/{DB_VERSION}/RECORDS-numerics"

SCREEN_PATH   = "screening_checkpoint.parquet"
RAW_DIR       = "data/raw_numerics"
MANIFEST_PATH = "data/download_manifest.parquet"

KEEP         = ["SPO2", "PULSE", "HR", "RESP", "ABPMEAN"]  
MAX_DROPOUT  = 0.3    # bỏ bệnh nhân có tỷ lệ mất SpO2 > 30% trong lần sàng lọc
SAVE_EVERY   = 25     # ghi manifest sau mỗi 25 record
MAX_RETRY    = 3

MAX_HOURS    = 72
N_PATIENTS   = 1000
SEED         = 42     # chọn bệnh nhân ngẫu nhiên, tái lập được
N_WORKERS    = 6

os.makedirs(RAW_DIR, exist_ok=True)
socket.setdefaulttimeout(60)

In [ ]:
t0 = time.time()
print("Đang tải danh sách record từ PhysioNet (1 request)...")
records = requests.get(RECORDS_URL, timeout=30).text.strip().splitlines()
print(f"  -> xong sau {time.time() - t0:.1f}s, {len(records)} record trong toàn DB")

patients = {}
for p in records:
    patients.setdefault(p.split("/")[1], []).append(p)

screen = pd.read_parquet(SCREEN_PATH)
good = screen.loc[screen["dropout_ratio_spo2"] <= MAX_DROPOUT, "patient_id"]
good_pids = good.sample(n=min(N_PATIENTS, len(good)), random_state=SEED)      # D5: lấy ngẫu nhiên, không lấy N bệnh nhân đầu file
todo = [r for pid in good_pids for r in patients[pid]]

print(f"Bệnh nhân hợp lệ: {len(good)} / {len(screen)} | chọn ngẫu nhiên {len(good_pids)} (seed {SEED})")
print(f"Tổng số record cần tải: {len(todo)}")

Đang tải danh sách record từ PhysioNet (1 request)...
  -> xong sau 69.7s, 22247 record trong toàn DB
Bệnh nhân hợp lệ: 1425 / 1605 | chọn ngẫu nhiên 1000 (seed 42)
Tổng số record cần tải: 2115


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_record(rec_path):
    pn_dir = f"{DB_NAME}/{DB_VERSION}/" + "/".join(rec_path.split("/")[:-1])
    name = rec_path.split("/")[-1]
    h = wfdb.rdheader(name, pn_dir=pn_dir)                      # request nhẹ
    names = [s.upper() for s in h.sig_name]
    if "SPO2" not in names or "PULSE" not in names:
        return None, h.fs                                       # bỏ qua, không tải dữ liệu
    sampto = min(h.sig_len, int(MAX_HOURS * 3600 * h.fs))
    rec = wfdb.rdrecord(name, pn_dir=pn_dir, sampto=sampto)
    df = pd.DataFrame(rec.p_signal, columns=[s.upper() for s in rec.sig_name])
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.reindex(columns=KEEP).replace(0, np.nan).astype("float32")
    return df, rec.fs

def process(rec_path):
    name = rec_path.split("/")[-1]
    row = {"record": rec_path, "patient_id": rec_path.split("/")[1], "status": "error",
           "fs": np.nan, "n_samples": 0, "hours": 0.0, "error": ""}
    t = time.time()
    for attempt in range(MAX_RETRY):
        try:
            df, fs = download_record(rec_path)
            row["fs"] = fs
            if df is None:
                row["status"] = "no_signal"
            else:
                df.to_parquet(f"{RAW_DIR}/{name}.parquet", index=False)
                row.update(status="ok", n_samples=len(df), hours=len(df) / fs / 3600,
                           **{f"cov_{c}": float(df[c].notna().mean()) for c in KEEP})
            row["error"] = ""
            break
        except Exception as e:
            row["error"] = f"{type(e).__name__}: {e}"[:200]
            time.sleep(2 * (attempt + 1))
    row["sec"] = time.time() - t
    return row

def save_manifest(rows):
    pd.DataFrame(list(rows.values())).to_parquet(MANIFEST_PATH, index=False)

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600:d}h{(sec % 3600) // 60:02d}m{sec % 60:02d}s"

# --- resume ---
if os.path.exists(MANIFEST_PATH):
    rows = {r["record"]: r for r in pd.read_parquet(MANIFEST_PATH).to_dict("records")}
    print(f"Resume: đã có {len(rows)} record trong manifest.")
else:
    rows = {}
done = {k for k, v in rows.items() if v["status"] in ("ok", "no_signal")}
remaining = [r for r in todo if r not in done]
total = len(remaining)
print(f"Đã xong: {len(done)} | Còn phải tải: {total} / {len(todo)} record")

# --- tải song song ---
n_ok = n_nosig = n_err = 0
t_start = time.time()
ex = ThreadPoolExecutor(max_workers=N_WORKERS)
try:
    futs = [ex.submit(process, r) for r in remaining]
    pbar = tqdm(as_completed(futs), total=total, desc="Download", unit="rec")
    for i, f in enumerate(pbar):
        row = f.result()
        rows[row["record"]] = row
        n_ok += row["status"] == "ok"
        n_nosig += row["status"] == "no_signal"
        n_err += row["status"] == "error"
        pbar.set_postfix(ok=n_ok, nosig=n_nosig, err=n_err, left=total - i - 1,
                         last=f"{row['sec']:.1f}s")
        if (i + 1) % SAVE_EVERY == 0:
            save_manifest(rows)
            el = time.time() - t_start
            eta = (total - i - 1) / ((i + 1) / el)
            tqdm.write(f"[{i + 1}/{total}] đã chạy {fmt(el)} | {(i + 1) / el * 60:.1f} rec/phút | "
                       f"ETA còn ~{fmt(eta)} | ok={n_ok} nosig={n_nosig} err={n_err}")
except KeyboardInterrupt:
    print("\n[Dừng thủ công] Hủy các tác vụ chưa chạy, đang lưu checkpoint...")
    ex.shutdown(wait=False, cancel_futures=True)
finally:
    save_manifest(rows)
    print(f"Đã lưu manifest: {len(rows)} record | tổng thời gian phiên này: {fmt(time.time() - t_start)}")

Resume: đã có 1667 record trong manifest.
Đã xong: 1667 | Còn phải tải: 945 / 2115 record


Download:   0%|          | 0/945 [00:00<?, ?rec/s]

In [ ]:
m = pd.read_parquet(MANIFEST_PATH)
ok = m[m["status"] == "ok"]

print(m["status"].value_counts().to_string(), "\n")
print(f"Record ok: {len(ok)} | Bệnh nhân: {ok['patient_id'].nunique()} | Tổng giờ: {ok['hours'].sum():,.0f}")
print(f"Tần số lấy mẫu (fs) duy nhất: {ok['fs'].unique()}")
print(f"Thời lượng record (giờ):\n{ok['hours'].describe().round(2).to_string()}\n")
print("Tỷ lệ có dữ liệu theo kênh (trung bình):")
print(ok[[f"cov_{c}" for c in KEEP]].mean().round(3).to_string())

missing = [r for r in ok["record"] if not os.path.exists(f"{RAW_DIR}/{r.split('/')[-1]}.parquet")]
print(f"\nFile bị thiếu trên đĩa: {len(missing)}")
print("Lỗi còn tồn đọng:", (m["status"] == "error").sum())

sample = pd.read_parquet(f"{RAW_DIR}/{ok.iloc[0]['record'].split('/')[-1]}.parquet")
display(sample.describe().round(1))

status
ok           1426
no_signal     241 

Record ok: 1426 | Bệnh nhân: 805 | Tổng giờ: 51,898
Tần số lấy mẫu (fs) duy nhất: [0.01666667 1.         0.01666667]
Thời lượng record (giờ):
count    1426.00
mean       36.39
std        27.80
min         0.01
25%        15.32
50%        28.17
75%        66.50
max       241.15

Tỷ lệ có dữ liệu theo kênh (trung bình):
cov_SPO2       0.851
cov_PULSE      0.817
cov_HR         0.942
cov_RESP       0.907
cov_ABPMEAN    0.168

File bị thiếu trên đĩa: 0
Lỗi còn tồn đọng: 0


,SPO2,PULSE,HR,RESP,ABPMEAN
count,14252.0,13062.0,14399.0,14395.0,13320.0
mean,96.5,63.9,64.0,23.1,79.4
std,2.8,10.5,10.6,6.6,28.0
min,15.4,42.2,41.6,8.3,-13.0
25%,94.8,57.3,57.5,18.8,65.5
50%,96.3,61.9,62.0,22.3,75.1
75%,99.0,68.1,68.2,28.0,85.8
max,100.0,117.2,121.2,48.9,354.0
